# Fairness Pipeline Development Toolkit — Demo

**Turing College AI Ethics Specialization** | Module 4, Sprint 1 | FairML Consulting

---

## Introduction

This notebook demonstrates the **Fairness Pipeline Development Toolkit** — an integrated system that orchestrates fairness across the pre-deployment ML lifecycle. The toolkit unifies three specialized modules (Measurement, Pipeline, Training) into a single, configuration-driven workflow.

### Problem Statement

Our client's data science teams face a recurring challenge: fairness interventions are applied **inconsistently and in isolation**. Individual modules exist for bias detection, data debiasing, and fair training — but without coordination, they fail to deliver systematic, reproducible fairness guarantees at scale.

In fairness literature, this is known as the **"pipeline leakage" problem**: even if one stage is debiased, downstream stages can reintroduce or amplify bias. A holistic approach that addresses fairness at every stage — data, preprocessing, training, and evaluation — is essential for trustworthy ML systems.

### Solution

This toolkit provides:
1. **Declarative configuration** (`config.yml`) — define the entire workflow in YAML
2. **Pipeline orchestration** (`run_pipeline.py`) — three-step automated pipeline
3. **MLflow integration** — experiment tracking for full traceability

### Outline

| Section | Description |
|---------|-------------|
| **1. Setup & Data Loading** | Load the German Credit dataset and train a baseline model |
| **2. Module 1: Measurement** | Compute fairness metrics with bootstrap CIs and reports |
| **3. Module 2: Pipeline** | Audit data for bias and apply pre-processing transformers |
| **4. Module 3: Training** | Train fair models using constrained optimization and regularization |
| **5. Integration: 3-Step Pipeline** | Execute the full orchestrated pipeline via `run_pipeline.py` |
| **6. Key Insights** | Discuss fairness-accuracy trade-offs and integration lessons |

---

## 1. Setup & Data Loading

Before any fairness analysis can begin, we need to prepare the environment and establish a **baseline model** — that is, a standard logistic regression trained without any fairness constraints. This baseline serves as the reference point against which all subsequent fairness interventions will be compared.

We use the **German Credit Dataset** (UCI / OpenML), a canonical benchmark in algorithmic fairness research. It contains 1,000 loan applicants classified as good or bad credit risks, with demographic attributes (sex, age) that serve as protected characteristics. This dataset is widely used because it exhibits real-world demographic disparities that mirror those found in production credit-scoring systems.

The following cell imports all required libraries and configures the environment.

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP
# =============================================================================
# This cell imports all third-party dependencies and ensures the fairness_toolkit
# package is importable from the current working directory. We suppress
# FutureWarnings to keep the notebook output clean — these warnings come from
# upstream libraries (sklearn, fairlearn) and are not relevant to our analysis.
# =============================================================================

import sys
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Suppress FutureWarnings from sklearn and fairlearn internals
warnings.filterwarnings('ignore', category=FutureWarning)

# Add the project root to sys.path so that `fairness_toolkit` is importable
# regardless of how the notebook kernel was launched
sys.path.insert(0, os.path.abspath('.'))

print('Environment ready.')

Next, we load the German Credit dataset using the toolkit's built-in loader. The loader handles fetching the data from OpenML, encoding categorical features, binarizing the target, and splitting into train/test sets. Importantly, it also separates the **sensitive features** (e.g., sex) from the main feature matrix — a key requirement for fairness-aware processing, since many fairness methods need the protected attribute as a separate input.

In [ ]:
# =============================================================================
# DATA LOADING AND TRAIN/TEST SPLIT
# =============================================================================
# We use the toolkit's `load_german_credit` function, which returns a dictionary
# containing pre-split arrays. The 70/30 split with a fixed random seed ensures
# reproducibility — critical for fairness evaluations where small changes in
# the test set composition can shift metric values significantly.
#
# The sensitive attribute 'sex' is extracted separately because fairness metrics
# require group membership labels alongside predictions. In a real-world setting,
# this attribute would typically be excluded from the feature matrix used for
# training ("fairness through unawareness"), though this alone is insufficient
# because proxy variables can encode the same information.
# =============================================================================

from fairness_toolkit.data.loader import load_german_credit

# Load and split the German Credit dataset
data = load_german_credit(test_size=0.3, random_state=42)

# Unpack the data dictionary into individual variables for clarity
X_train = data['X_train']       # Feature matrix for training (numeric, no sensitive cols)
X_test = data['X_test']         # Feature matrix for evaluation
y_train = data['y_train']       # Binary target: 1 = good credit, 0 = bad credit
y_test = data['y_test']
sens_train = data['sensitive_train']  # DataFrame with protected attribute columns
sens_test = data['sensitive_test']

# Print dataset summary to verify the loading was successful
print(f'Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Sensitive columns: {data["sensitive_columns"]}')
print(f'Target: {data["target_name"]} (positive = {data["positive_label"]})')

# Display the group distribution in the test set.
# An imbalanced distribution (e.g., 70% male / 30% female) is itself a form
# of representation bias that the BiasDetectionEngine will flag later.
print(f'\nSensitive attribute distribution (test set):')
print(sens_test['sex'].value_counts())

Now we train the **baseline model** — a standard logistic regression with no fairness constraints. This model represents what a typical ML practitioner would deploy without considering fairness. Its accuracy and fairness metrics will serve as the "before" snapshot in our before/after comparison.

Logistic regression is chosen deliberately: it is interpretable, fast to train, and commonly used in credit scoring. The fairness disparities we observe here are not artifacts of model complexity — they arise from genuine patterns of historical discrimination encoded in the data.

In [ ]:
# =============================================================================
# BASELINE (UNCONSTRAINED) MODEL
# =============================================================================
# This cell trains a vanilla logistic regression — our "unfair" baseline.
# The purpose is twofold:
#   1. Establish the accuracy ceiling — any fairness intervention will likely
#      reduce accuracy somewhat (the fairness-accuracy trade-off).
#   2. Measure the initial level of bias so we can quantify how much our
#      interventions improve fairness.
#
# max_iter=1000 ensures convergence; the default (100) can fail on this dataset.
# =============================================================================

# Train a baseline (unfair) model
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_preds)

print(f'Baseline model accuracy: {baseline_acc:.4f}')
print(f'\nClassification Report:')
# The per-class precision/recall breakdown helps us understand whether the model
# is biased toward predicting "Good Credit" (the majority class) at the expense
# of correctly identifying bad credit risks.
print(classification_report(y_test, baseline_preds, target_names=['Bad Credit', 'Good Credit']))

---

## 2. Module 1: Measurement

The **MeasurementModule** provides the `FairnessAnalyzer` class — a unified interface for computing fairness metrics with bootstrap confidence intervals, effect sizes (risk ratios), and human-readable reports.

Measurement is the foundation of any fairness workflow. Without rigorous, statistically grounded metrics, it is impossible to know whether a model is biased, how severe the bias is, or whether an intervention actually improved things. This module addresses these needs.

### Key Components
- **Demographic Parity Difference (DPD)**: The maximum gap in positive prediction rates across demographic groups. A DPD of 0 means all groups receive positive predictions at equal rates — regardless of the true outcome distribution. This is the simplest fairness criterion but can conflict with predictive accuracy when base rates differ across groups.
- **Equalized Odds Difference (EOD)**: The maximum of true positive rate (TPR) and false positive rate (FPR) disparities across groups. Unlike demographic parity, equalized odds conditions on the true label, asking that the model be equally accurate for all groups.
- **Bootstrap 95% CI**: Resampling-based confidence intervals provide statistical rigor — a point estimate of 0.08 DPD could easily be noise on a small test set. CIs help distinguish genuine bias from sampling variability.
- **Risk Ratio**: A disparate impact measure where 1.0 indicates parity. In U.S. employment law, a ratio below 0.8 (the "four-fifths rule") is considered evidence of adverse impact.

The following cell creates a `FairnessAnalyzer` for the baseline model and computes both DPD and EOD with bootstrap confidence intervals.

In [ ]:
# =============================================================================
# FAIRNESS MEASUREMENT WITH BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================
# The FairnessAnalyzer wraps Fairlearn's MetricFrame and adds bootstrap CIs
# and effect sizes on top. We compute two complementary fairness metrics:
#
# - demographic_parity_difference: Measures whether positive predictions are
#   distributed equally across groups, irrespective of true labels. This captures
#   "selection rate" fairness.
#
# - equalized_odds_difference: Measures whether the model's error rates (TPR, FPR)
#   are equal across groups. This captures "error rate" fairness.
#
# Using 500 bootstrap samples provides a good balance between CI precision and
# computation time. For production audits, 1000+ samples would be recommended.
# =============================================================================

from fairness_toolkit.measurement.analyzer import FairnessAnalyzer

# Create analyzer for the baseline model
analyzer = FairnessAnalyzer(
    y_true=y_test,
    y_pred=baseline_preds,
    sensitive_features=sens_test['sex'].values,
)

# Compute metrics with bootstrap CIs
results = analyzer.compute_metrics(
    metrics=['demographic_parity_difference', 'equalized_odds_difference'],
    n_bootstrap=500,
)

# Display results — each metric includes its point estimate, 95% CI, effect size,
# and group sizes. The CI is critical: if it includes 0, we cannot confidently
# claim that bias exists. If the entire CI is above the threshold, the bias is
# statistically significant.
for metric_name, metric_data in results.items():
    print(f'\n{metric_name}:')
    print(f'  Value:       {metric_data["value"]:.4f}')
    print(f'  95% CI:      [{metric_data["ci_lower"]:.4f}, {metric_data["ci_upper"]:.4f}]')
    print(f'  Effect Size: {metric_data["effect_size"]:.4f}')  # Risk ratio: <0.8 triggers four-fifths rule
    print(f'  Group Sizes: {metric_data["group_sizes"]}')

The `FairnessAnalyzer` also produces a **human-readable report** that summarizes the findings in plain language. This is designed for non-technical stakeholders (compliance officers, product managers) who need to understand the fairness implications without interpreting raw numbers.

In [ ]:
# =============================================================================
# HUMAN-READABLE FAIRNESS REPORT
# =============================================================================
# The generate_report() method produces a structured text summary that translates
# the numeric results into actionable insights. This bridges the gap between
# data scientists (who work with metrics) and decision-makers (who need to
# understand the business and ethical implications).
#
# In production, this report would typically be attached to model approval
# documents or compliance records.
# =============================================================================

# Generate human-readable report
report = analyzer.generate_report()
print(report)

Finally, the Measurement module includes a **pytest-compatible assertion** (`assert_fairness`) that can be embedded in CI/CD pipelines to prevent biased models from reaching production. If the fairness metric exceeds the specified threshold, the assertion fails and blocks deployment.

Here we deliberately use a strict threshold (0.05) that we expect the baseline model to violate, demonstrating the gating mechanism.

In [ ]:
# =============================================================================
# CI/CD FAIRNESS GATE: assert_fairness()
# =============================================================================
# This function is designed for use in automated test suites (pytest). It raises
# an AssertionError if the specified fairness metric exceeds the threshold.
#
# In practice, this would be called in a GitHub Actions workflow (see
# .github/workflows/fairness_check.yml) to enforce fairness constraints as a
# deployment gate. A model that fails this check would not be promoted to
# production, forcing the team to apply mitigation strategies first.
#
# We use a strict threshold of 0.05 here to demonstrate the failure case.
# The baseline model (trained without constraints) is expected to exceed this.
# =============================================================================

# Demonstrate the pytest-compatible fairness assertion
from fairness_toolkit.measurement.integrations import assert_fairness

try:
    assert_fairness(
        y_test, baseline_preds, sens_test['sex'].values,
        metric='demographic_parity_difference',
        threshold=0.05,  # Strict threshold — most unconstrained models will fail this
    )
    print('Fairness check PASSED')
except AssertionError as e:
    print(f'Fairness check result: {e}')
    print('\nThis is expected — the baseline model has not been debiased yet.')

---

## 3. Module 2: Pipeline (Data Engineering)

The **PipelineModule** addresses fairness at the **data level** — before any model is trained. This reflects a key insight from fairness research: bias in training data is the single largest source of unfair model behavior. Even a perfectly specified model will learn discriminatory patterns if the data encodes historical prejudice.

The module provides two capabilities:
1. **BiasDetectionEngine** — audits raw data for representation bias (group imbalance), statistical disparity (outcome rate differences), and proxy variables (features highly correlated with the protected attribute)
2. **Pre-processing transformers** — sklearn-compatible transformers that mitigate bias before model training

### Available Transformers
| Transformer | Strategy | How it works |
|-------------|----------|------------|
| `DisparateImpactRemover` | Feature repair | Shifts group-conditional feature distributions toward the overall distribution, reducing the predictive power of protected group membership |
| `InstanceReweighter` | Sample reweighting | Assigns higher weights to underrepresented group-outcome combinations, effectively balancing the training signal across groups |

The following cell runs a comprehensive bias audit on the training data to establish what forms of bias exist before any mitigation.

In [ ]:
# =============================================================================
# BIAS AUDIT: BiasDetectionEngine
# =============================================================================
# The BiasDetectionEngine performs three types of analysis:
#
# 1. Representation Bias: Uses a chi-squared test to determine whether group
#    proportions deviate significantly from uniform. In the German Credit dataset,
#    males are overrepresented (~69%), which can cause models to learn male-centric
#    decision boundaries.
#
# 2. Statistical Disparity: Tests whether outcome rates (good credit vs bad credit)
#    differ significantly across groups using chi-squared tests per feature.
#
# 3. Proxy Detection: Identifies features that are highly correlated with the
#    protected attribute. Even if we remove 'sex' from training, proxy variables
#    (e.g., certain job categories) can still encode group membership, making
#    "fairness through unawareness" ineffective.
# =============================================================================

from fairness_toolkit.pipeline.detection import BiasDetectionEngine

# Create a DataFrame for the bias audit — the engine expects a single DataFrame
# with the sensitive column and target included alongside features
audit_df = pd.DataFrame(X_train, columns=data['feature_names'])
audit_df['sex'] = sens_train['sex'].values
audit_df['target'] = y_train

# Run a full bias audit across all three dimensions
engine = BiasDetectionEngine(sensitive_column='sex')
audit_report = engine.full_audit(audit_df, target_column='target')

# Display summary — this gives a quick overview of the bias landscape
print('=== BIAS AUDIT SUMMARY ===')
summary = audit_report['summary']
print(f'  Representation bias detected: {summary["has_representation_bias"]}')
print(f'  Proxy variables found:        {summary["proxy_count"]}')
print(f'  Significant disparities:       {summary["significant_disparity_count"]}')

# Show representation bias details — this tells us whether the dataset has
# balanced group representation. A significant p-value means the groups are
# not equally represented, which can bias model training.
rep = audit_report['representation_bias']
print(f'\n=== REPRESENTATION BIAS ===')
print(f'  Group counts:      {rep["group_counts"]}')
print(f'  Group proportions: {rep["group_proportions"]}')
print(f'  Chi-squared stat:  {rep["chi2_statistic"]:.4f}')  # Higher = more imbalanced
print(f'  p-value:           {rep["p_value"]:.4f}')          # <0.05 = statistically significant imbalance

Now we apply the **DisparateImpactRemover**, a pre-processing transformer inspired by Feldman et al. (2015). This transformer works by adjusting the feature distributions within each demographic group so that they become more similar to the overall population distribution. The `repair_level` parameter (0.0 to 1.0) controls how aggressively this adjustment is applied:

- `repair_level=0.0`: No change (original data)
- `repair_level=1.0`: Full repair (group distributions become identical)
- `repair_level=0.8`: Our choice — a strong repair that preserves some feature variance

Note the data flow: the sensitive attribute must be temporarily appended to the feature matrix for the transformer to identify group membership, then removed before model training.

In [ ]:
# =============================================================================
# PRE-PROCESSING: DisparateImpactRemover
# =============================================================================
# The DisparateImpactRemover modifies feature values so that they carry less
# information about group membership. Conceptually, it asks: "What would these
# feature values look like if group membership were independent of feature
# distributions?" — and then moves the data toward that counterfactual.
#
# Implementation detail: the transformer requires the sensitive attribute to be
# present as a column in the feature matrix (to identify groups), but we remove
# it after transformation. This is a common pattern in fairness preprocessing.
#
# The repair_level of 0.8 is a pragmatic choice: full repair (1.0) can destroy
# too much predictive signal, while low repair (e.g., 0.3) may leave substantial
# bias intact. 0.8 is a commonly used default in the literature.
# =============================================================================

from fairness_toolkit.pipeline.transformers import DisparateImpactRemover

# Convert the categorical sensitive attribute to numeric (male=1, female=0)
# because the transformer operates on numeric arrays
sens_numeric_train = (sens_train['sex'] == 'male').astype(float).values
sens_numeric_test = (sens_test['sex'] == 'male').astype(float).values

# Record the index where the sensitive column will be appended
sens_col_idx = X_train.shape[1]

# Augment the feature matrix by appending the sensitive attribute as the last column
X_train_aug = np.column_stack([X_train, sens_numeric_train])
X_test_aug = np.column_stack([X_test, sens_numeric_test])

# Fit the transformer on training data and apply to both train and test
# (fit on train only to avoid data leakage)
remover = DisparateImpactRemover(sensitive_column_index=sens_col_idx, repair_level=0.8)
remover.fit(X_train_aug)
X_train_repaired = remover.transform(X_train_aug)
X_test_repaired = remover.transform(X_test_aug)

# Remove the sensitive column — it was only needed for group identification
# during transformation and should NOT be used as a training feature
X_train_clean = X_train_repaired[:, :sens_col_idx]
X_test_clean = X_test_repaired[:, :sens_col_idx]

print(f'Original feature matrix shape:   {X_train.shape}')
print(f'Repaired feature matrix shape:   {X_train_clean.shape}')
print(f'Repair level: 0.8')
print(f'\nDisparate Impact Remover applied successfully.')

---

## 4. Module 3: Training (Fair Model Training)

Even after pre-processing, the training process itself can introduce or amplify bias. The **TrainingModule** provides multiple strategies for incorporating fairness constraints directly into the learning algorithm. These represent the three major paradigms in fair ML training:

| Method | Approach | Framework | When to use |
|--------|----------|-----------|-------------|
| `ReductionsWrapper` | Constrained optimization via exponentiated gradient | sklearn + Fairlearn | Production sklearn pipelines; strongest theoretical guarantees |
| `FairnessRegularizer` | Regularized loss: BCE + eta * penalty | PyTorch | Custom neural network training; flexible penalty tuning |
| `GroupFairnessCalibrator` | Post-processing per-group calibration | sklearn | When retraining is not possible; adjusts an existing model's outputs |

The following cells demonstrate each approach in turn.

### 4.1 Constrained Optimization (ReductionsWrapper)

The `ReductionsWrapper` implements the exponentiated gradient method from Agarwal et al. (2018), which reduces fair classification to a sequence of cost-sensitive classification problems. The `eps` parameter controls how tight the fairness constraint is — smaller values mean stricter fairness but potentially more accuracy loss.

In [ ]:
# =============================================================================
# FAIR TRAINING: ReductionsWrapper (Exponentiated Gradient)
# =============================================================================
# The ReductionsWrapper uses Fairlearn's ExponentiatedGradient algorithm, which
# solves the constrained optimization problem:
#     minimize   classification_error(model)
#     subject to fairness_violation(model) <= eps
#
# The algorithm works by iteratively adjusting sample weights to find a
# randomized classifier that satisfies the constraint. eps=0.01 is very tight,
# meaning we demand near-perfect demographic parity — this will likely cost
# some accuracy compared to the unconstrained baseline.
#
# We train on the pre-processed (repaired) data from Module 2, which means
# the fair training builds on top of the debiased features — a layered defense
# approach that is more robust than either intervention alone.
# =============================================================================

from fairness_toolkit.training.reductions import ReductionsWrapper
from fairness_toolkit.measurement.metrics import demographic_parity_difference

# Train a fair model using ReductionsWrapper with demographic parity constraint
wrapper = ReductionsWrapper(
    estimator=LogisticRegression(max_iter=1000),
    constraint='demographic_parity',
    eps=0.01,  # Very tight constraint: allows at most 1% DPD
)

# The sensitive_features argument tells the algorithm which groups to equalize.
# Note: we pass the original categorical labels, not the numeric encoding.
wrapper.fit(
    X_train_clean,
    y_train,
    sensitive_features=sens_train['sex'].values,
)

fair_preds = wrapper.predict(X_test_clean)
fair_acc = accuracy_score(y_test, fair_preds)

# Compare baseline vs fair model to quantify the trade-off
baseline_dpd = demographic_parity_difference(baseline_preds, sens_test['sex'].values)
fair_dpd = demographic_parity_difference(fair_preds, sens_test['sex'].values)

print('=== BASELINE vs FAIR MODEL ===')
print(f'\n  Baseline:')
print(f'    Accuracy: {baseline_acc:.4f}')
print(f'    DPD:      {baseline_dpd:.4f}')
print(f'\n  Fair Model (ReductionsWrapper):')
print(f'    Accuracy: {fair_acc:.4f}')
print(f'    DPD:      {fair_dpd:.4f}')
# The "improvement" section quantifies the fairness-accuracy trade-off:
# a positive DPD reduction means improved fairness; a positive accuracy cost
# means we paid for that fairness with some predictive performance.
print(f'\n  Improvement:')
print(f'    DPD reduction: {baseline_dpd - fair_dpd:+.4f}')
print(f'    Accuracy cost: {baseline_acc - fair_acc:+.4f}')

### 4.2 Fairness Regularization (PyTorch)

The `FairnessRegularizer` adds a differentiable fairness penalty to the standard binary cross-entropy (BCE) loss. The `eta` parameter controls the trade-off:

```
Total Loss = BCE(predictions, targets) + eta * Fairness_Penalty(predictions, groups)
```

As `eta` increases, the optimizer allocates more effort to satisfying the fairness constraint, potentially at the cost of raw predictive accuracy. This is the neural network analog of the ReductionsWrapper's `eps` parameter.

The cell below demonstrates how different values of `eta` shift the balance between prediction accuracy (BCE) and fairness (penalty).

In [ ]:
# =============================================================================
# FAIRNESS REGULARIZATION: FairnessRegularizer (PyTorch)
# =============================================================================
# This cell demonstrates the FairnessRegularizer in isolation, using synthetic
# data to show how the eta parameter controls the fairness-accuracy trade-off.
#
# The synthetic setup is deliberately biased: Group 0 receives high logits
# (strong positive predictions) while Group 1 receives low logits (strong
# negative predictions). This creates a large demographic parity gap that the
# regularizer penalizes.
#
# Key insight: at eta=0, the loss is pure BCE (no fairness consideration).
# As eta increases, the penalty term dominates, forcing the model to equalize
# prediction rates across groups during gradient descent.
# =============================================================================

import torch
from fairness_toolkit.training.regularizer import FairnessRegularizer

# Demonstrate the PyTorch FairnessRegularizer
print('=== FairnessRegularizer Demo ===')
print('Loss = BCE + eta * fairness_penalty\n')

# Simulate biased predictions: Group 0 gets uniformly high logits,
# Group 1 gets uniformly low logits — maximum demographic disparity
logits = torch.tensor([2.0, 1.5, 1.0, 0.5, -1.0, -1.5, -2.0, -0.5])
targets = torch.tensor([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
sensitive = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]).float()  # Group 0 gets high logits

# Sweep eta values to show the trade-off curve
for eta in [0.0, 0.5, 1.0, 5.0, 10.0]:
    reg = FairnessRegularizer(eta=eta)
    total, bce, penalty = reg(logits, targets, sensitive)
    # Notice: BCE stays constant (same predictions), but the total loss increases
    # as eta amplifies the fairness penalty
    print(f'  eta={eta:5.1f}  |  Total={total.item():.4f}  |  BCE={bce.item():.4f}  |  Penalty={penalty.item():.4f}')

print('\nAs eta increases, the fairness penalty has more influence on the total loss.')

### 4.3 Post-Processing: Group Calibration

The `GroupFairnessCalibrator` takes a different approach: rather than modifying the data or the training process, it adjusts the **model's output probabilities** after training. It fits a separate calibration function (Platt scaling or isotonic regression) for each demographic group, ensuring that predicted probabilities are well-calibrated within each group.

This is particularly useful when:
- Retraining the model is expensive or infeasible
- The model is a black box (e.g., a vendor-supplied scoring model)
- Calibration is needed for fair decision-making at different threshold values

The cell below shows how per-group calibration adjusts the probability distributions so that both groups receive similar mean predicted probabilities.

In [ ]:
# =============================================================================
# POST-PROCESSING: GroupFairnessCalibrator
# =============================================================================
# The GroupFairnessCalibrator fits separate Platt scaling models for each
# demographic group. Platt scaling transforms raw probabilities through a
# learned sigmoid function: P_calibrated = sigmoid(a * P_raw + b).
#
# By fitting separate (a, b) parameters per group, the calibrator can correct
# systematic over- or under-confidence that differs across groups. For example,
# if the model is systematically overconfident for male applicants, the male
# calibrator will learn to attenuate those probabilities.
#
# This is a post-hoc intervention — it does not change the model itself, only
# how its outputs are interpreted. It is the least invasive fairness strategy.
# =============================================================================

from fairness_toolkit.training.calibration import GroupFairnessCalibrator

# Extract predicted probabilities from the baseline model (probability of Good Credit)
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]

# Fit per-group Platt scaling calibrators
cal = GroupFairnessCalibrator(method='platt')
cal.fit(baseline_probs, y_test, sens_test['sex'].values)
calibrated_probs = cal.transform(baseline_probs, sens_test['sex'].values)

# Compare probability distributions before and after calibration.
# If calibration is working correctly, the mean predicted probabilities
# should become more similar across groups after calibration.
print('=== GroupFairnessCalibrator (Platt) ===')
print(f'\nBefore calibration:')
for group in ['male', 'female']:
    mask = sens_test['sex'].values == group
    print(f'  {group:8s} mean prob: {baseline_probs[mask].mean():.4f}  (std: {baseline_probs[mask].std():.4f})')

print(f'\nAfter calibration:')
for group in ['male', 'female']:
    mask = sens_test['sex'].values == group
    print(f'  {group:8s} mean prob: {calibrated_probs[mask].mean():.4f}  (std: {calibrated_probs[mask].std():.4f})')

---

## 5. Integration: 3-Step Pipeline

The core deliverable is the **orchestrator** (`run_pipeline.py`) which parses `config.yml` and executes the three-step workflow automatically. This section demonstrates the full integration.

The pipeline implements a principled three-stage approach:
1. **Baseline Measurement** — Quantify the current level of bias (how bad is it?)
2. **Transform & Train** — Apply data debiasing + fair training (fix it)
3. **Final Validation** — Re-measure and compare against threshold (did it work?)

This structure ensures that fairness is not just applied but also **verified** — the pipeline produces a clear PASS/FAIL verdict based on the configured threshold.

### Configuration

The pipeline behavior is entirely controlled by `config.yml`. This declarative approach means that different teams (data engineering, ML, compliance) can review and agree on the same specification without reading code. Let us inspect the current configuration.

In [ ]:
# =============================================================================
# CONFIGURATION DISPLAY
# =============================================================================
# The config.yml file is the single source of truth for the pipeline. It defines:
#   - Which dataset to use and how to split it
#   - Which sensitive attribute to protect
#   - Which fairness metrics to compute and how many bootstrap samples
#   - Which pre-processing transformer to apply (and with what parameters)
#   - Which training method to use (and with what constraints)
#   - What fairness threshold determines PASS vs FAIL
#   - Where to log results in MLflow
#
# This separation of configuration from code is essential for production ML
# fairness: it makes the fairness requirements auditable, version-controlled,
# and modifiable without code changes.
# =============================================================================

import yaml

# Load and display the configuration
with open('config.yml', 'r') as f:
    config = yaml.safe_load(f)

print('=== PIPELINE CONFIGURATION ===')
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

Now we execute the **full orchestrated pipeline**. The `main()` function in `run_pipeline.py` will:
1. Load the dataset and train a baseline model
2. Measure baseline fairness metrics with bootstrap CIs
3. Apply the DisparateImpactRemover to debias the features
4. Train a fair model using the ReductionsWrapper
5. Measure final fairness metrics and compare against the threshold
6. Log everything to MLflow (metrics, model artifact, config)
7. Return True if the pipeline PASSES, False if it FAILS

In [ ]:
# =============================================================================
# EXECUTE THE FULL 3-STEP PIPELINE
# =============================================================================
# This single call orchestrates the entire fairness workflow: measure, transform,
# train, validate, and log. It demonstrates the value of the integration layer —
# rather than manually coordinating three modules (as we did above), the
# orchestrator handles data flow, feature matrix augmentation/cleanup, and
# MLflow experiment management automatically.
#
# The returned boolean indicates whether the final model's fairness metric
# is at or below the configured threshold (PASS=True, FAIL=False).
# =============================================================================

# Execute the full pipeline
from run_pipeline import main

passed = main('config.yml')

After the pipeline runs, all results are persisted in MLflow. The cell below retrieves the most recent experiment run and displays the logged metrics. This demonstrates the traceability that MLflow provides — every pipeline execution creates a permanent, queryable record of what was measured, what thresholds were applied, and what the outcome was.

In [ ]:
# =============================================================================
# VERIFY MLFLOW EXPERIMENT TRACKING
# =============================================================================
# MLflow stores all pipeline results in a structured experiment store. Each run
# records:
#   - Metrics: accuracy, DPD, EOD (both baseline and final)
#   - Parameters: transformer type, repair level, constraint, eps, threshold
#   - Artifacts: the trained fair model, the config.yml used
#
# This is critical for regulatory compliance (e.g., EU AI Act, ECOA) where
# organizations must demonstrate that they have tested for and mitigated bias.
# The MLflow record serves as an audit trail.
# =============================================================================

# Verify MLflow logged the results
import mlflow

mlflow.set_tracking_uri('mlruns')

# Retrieve the most recent run from the pipeline experiment
experiment = mlflow.get_experiment_by_name('fairness_pipeline_run')
if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], max_results=1)
    if not runs.empty:
        print('=== MLFLOW LOGGED RESULTS ===')
        print(f'\nRun ID: {runs.iloc[0]["run_id"]}')
        print(f'Status: {runs.iloc[0]["status"]}')
        # Display all logged metrics — these capture the full before/after picture
        print(f'\nMetrics:')
        metric_cols = [c for c in runs.columns if c.startswith('metrics.')]
        for col in sorted(metric_cols):
            val = runs.iloc[0][col]
            if pd.notna(val):  # Skip metrics that were not logged in this run
                print(f'  {col.replace("metrics.", ""):40s} = {val:.4f}')
        print(f'\nArtifacts logged: config.yml, fair_model')
    else:
        print('No runs found.')
else:
    print('Experiment not found. Run the pipeline first.')

---

## 6. Key Insights

### Fairness-Accuracy Trade-off

The pipeline consistently demonstrates the fundamental tension in fairness-aware ML:

- **Stricter constraints** (`eps=0.01`) produce lower disparity but may reduce accuracy
- **Looser constraints** (`eps=0.05`) preserve more accuracy but allow more residual bias
- The `config.yml` threshold provides a **principled decision boundary** — the pipeline either PASSES or FAILS

This trade-off is not a bug — it is an inherent property of fairness-constrained optimization, formally characterized by Chouldechova (2017) and Kleinberg et al. (2016). When base rates differ across groups (as they often do in real data), it is mathematically impossible to simultaneously satisfy all fairness criteria while maintaining perfect accuracy. The toolkit makes this trade-off explicit and measurable rather than hidden.

### Integration Complexity

Integrating multiple fairness modules revealed several challenges:

1. **Data flow management**: The transformer modifies features that the training method then consumes. The orchestrator must carefully manage the augmented feature matrix (adding/removing the sensitive column at the right stages).

2. **Measurement consistency**: Using the same `FairnessAnalyzer` for both baseline and final evaluation ensures that metrics are computed identically, making the comparison valid.

3. **Configuration as contract**: The `config.yml` serves as a contract between teams — data engineers, model developers, and compliance officers can all read and agree on the same specification.

### Production Readiness

The toolkit demonstrates several production-ready patterns:

- **Declarative configuration** enables reproducibility and auditability
- **MLflow integration** provides full experiment traceability
- **CI/CD fairness gates** (`assert_fairness`) prevent biased models from reaching production
- **Bootstrap confidence intervals** add statistical rigor beyond point estimates

### Adaptability

While demonstrated with the German Credit dataset, the toolkit can be adapted for:

- **Different datasets**: Any binary classification task with protected attributes
- **Different constraints**: Swap `demographic_parity` for `equalized_odds` in the config
- **Different transformers**: Switch between `DisparateImpactRemover` and `InstanceReweighter`
- **Different industries**: Healthcare, hiring, insurance, criminal justice — anywhere algorithmic fairness matters